# 03 - Introduction to Agents

In the first section, we built an **AI Travel Assistant** — a basic LLM app that can answer travel-related questions.

### Answers, but no actions

The initial assistant can generate helpful responses, but it cannot act.  
It is limited to the prompt and Its training data  

It cannot check the weather, search for flights or convert currencies  

### Moving towards agents

To make the assistant more useful, we need it to go beyond text generation.

We want it to:
- Use external tools  
- Retrieve real-time information  
- Perform tasks  

This is where **agents** come in.

### LLM app vs Agent

A basic LLM app:  

``` Takes a prompt``` → ```returns an answer```

An agent adds a loop:

``` User``` → ```LLM``` → ```Tool``` → ```LLM``` →```Response```

An **agent = LLM + tools**, you can think of it as brains having hands

## Setup

Before running the below cell, ensure you have:

1. Authenticated with `gcloud auth application-default login`
2. Set your GCP project and location below


In [ ]:
import os
import json
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path().resolve().parents[1] / ".env")

In [ ]:
# This workshop uses the `google-genai` Python package with Vertex AI.
from google import genai

# Set GCP project and location
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
location = os.getenv("GOOGLE_CLOUD_LOCATION")

# Initialize the genai client for Vertex AI
client = genai.Client(
    vertexai=True, 
    project=project_id, 
    location=location
)

In [ ]:
from google.genai.types import GenerateContentConfig

# Select the model
MODEL_NAME = "gemini-2.5-flash"
# Set a default system message for the model
SYSTEM_MESSAGE = """
    You are a helpful travel assistant.
    You have access to tools. Decide whether you need to use a tool.
    - If needed, use it
    - Otherwise, answer directly
"""

def ask_llm(
    prompt: str,
    system_instruction: str = SYSTEM_MESSAGE,
    temperature: float = 0.7,
    top_k: int = 40,
    top_p: float = 1.0,
    tools: list = None,  # ✅ NEW
) -> str:
    """Send a prompt to the LLM and return the text response."""

    config = GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        tools=tools,  # <-- tools are passed here
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config,
    )

    return response.text

## First tool call

**What is a tool?**

A tool is just a python function the LLM can call for a specific task.

For this workshop, we use tools that read local mock data instead of calling real weather, flight, or exchange-rate APIs.

#### Load mock tool data

These files act like tiny fake APIs. They make the workshop repeatable because the data does not change.

In [ ]:
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")

with open(DATA_DIR / "mock_weather.json", "r", encoding="utf-8") as file:
    weather_data = json.load(file)

with open(DATA_DIR / "mock_flights.json", "r", encoding="utf-8") as file:
    flight_data = json.load(file)

print("Mock data loaded.")

#### Create mock tools

Each tool is a normal Python function. The model does not call these directly yet.

In [ ]:
def get_weather(location: str) -> dict:
    """Returns the current weather for a location.
    Args:
        location: The city name, for example: Lisbon
    """
    print(f"🔧 TOOL CALLED: get_weather(location={location})")
    return weather_data.get(location,{"error": f"No mock weather found for {location}."})


def search_flights(origin: str, destination: str) -> list:
    """Return mock flights for an origin-destination pair."
     Args:
        origin: The departure city, for example: Lisbon
        destination: The arrival city, for example: Paris
    """
    print(f"🔧 TOOL CALLED: search_flights(origin={origin}, destination={destination})")
    route = f"{origin}-{destination}"
    return flight_data.get(route, [])


def your_tool():
    """Define your own tool here!"""
    pass

In [ ]:
prompt = """
I am visiting Lisbon this weekend. What should I pack?
"""

response = ask_llm(
    prompt,
    tools=[get_weather],  # ✅ pass tool here
)

print(f"\nResponse:\n{response}")

## Multi-tool workflow

Now we ask a question that may require more than one tool.

The assistant needs to:
- Search for a flight
- Check company travel policy
- Possibly convert currency or explain cost


This is closer to an agent workflow.

The assistant may need to combine information from multiple tools before giving an answer.


In [ ]:
prompt = """
I am planning a weekend work trip to Lisbon.
Please help me decide what to pack, and whether a flight is available.
I will be departing from Amsterdam.
"""

response = ask_llm(
    prompt,
    tools=[get_weather, search_flights],
)

print(f"\nResponse:\n{response}")

## Exercise: Add Your Own Tool

Now it's your turn to extend the agent!

Create a new tool function. Your tool should take input parameters and return a mock result.

Make sure to update the prompt* so that the user's question clearly requires your tool (e.g., "How much should I budget per day in Paris for a student?").

**Example Ideas:**
- `estimate_daily_budget(city, style)`
- `exchange_rate_calculator(from_currency, to_currency)`
- `suggest_airport_transfer(city)`

Try it out and see how the LLM responds when you provide tool results!